<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Gut_Microbiome_Metagenomics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Gut Microbiome Metagenomics Analysis

Project 14 — Advanced Bioinformatics

Is notebook mein hum **gut microbiome 16S rRNA taxonomic composition data** par ek **complete metagenomics analysis pipeline** run karain gay — Healthy vs Disease (IBD-like) cohort compare karte hue — diversity analysis, ordination, differential abundance testing, aur ML-based classification ke sath.

**Data approach:** Hum **EBI MGnify** (real public metagenomics data repository) ke REST API se real taxonomic abundance data fetch karne ki koshish karte hain. Agar live API call fail ho (network/endpoint change), notebook automatically ek **literature-grounded realistic dataset** use karta hai — genus-level abundance patterns jo real published gut microbiome studies (Firmicutes/Bacteroidetes ratio shifts, reduced *Faecalibacterium* aur increased *Proteobacteria* in IBD, etc.) ko accurately reflect karte hain — taake pipeline hamesha meaningful results de.

---

## 📋 Table of Contents

| Section | Content |
|---|---|
| 1 | Setup & Installation |
| 2 | Data Acquisition (MGnify API + Fallback) |
| 3 | Data Preprocessing & Relative Abundance |
| 4 | Taxonomic Composition Overview |
| 5 | Alpha Diversity (Shannon, Simpson, Richness) |
| 6 | Beta Diversity & PCoA Ordination |
| 7 | Differential Abundance Testing |
| 8 | Unsupervised Community Clustering (Enterotyping) |
| 9 | ML Classification — Healthy vs Disease |
| 10 |  **Runtime Prediction** — Apna Microbiome Profile Daal Kar Predict Karein |


## 1. Setup & Installation

In [1]:
!pip install -q scikit-bio plotly scikit-learn scipy pandas numpy ipywidgets requests

import numpy as np
import pandas as pd
import requests
import warnings
warnings.filterwarnings("ignore")

from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.metrics import silhouette_score

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.8 MB/s eta 0:00:00


## 2. Data Acquisition — Real Gut Microbiome Data

**Primary path:** EBI MGnify public REST API (ebi.ac.uk/metagenomics) se real study taxonomic summaries fetch karna.
**Fallback path:** Agar API unavailable ho, ek literature-grounded genus-level abundance table generate karta hai jo real published Healthy vs IBD gut microbiome studies ke taxonomic shift patterns follow karta hai.


In [2]:
MGNIFY_BASE = "https://www.ebi.ac.uk/metagenomics/api/v1"

def fetch_mgnify_study_taxonomy(study_accession, max_analyses=50):
    """Attempt to fetch real taxonomic abundance summaries for a public MGnify study."""
    resp = requests.get(f"{MGNIFY_BASE}/studies/{study_accession}/analyses",
                         params={"page_size": max_analyses}, timeout=30)
    resp.raise_for_status()
    analyses = resp.json()["data"]

    records = []
    for a in analyses:
        acc = a["id"]
        tax_resp = requests.get(f"{MGNIFY_BASE}/analyses/{acc}/taxonomy", timeout=30)
        if tax_resp.status_code != 200:
            continue
        tax_data = tax_resp.json()["data"]
        for entry in tax_data:
            attrs = entry.get("attributes", {})
            records.append({"sample": acc, "taxon": attrs.get("lineage", "Unknown"), "count": attrs.get("count", 0)})
    return pd.DataFrame(records)

try:
    raw_taxonomy = fetch_mgnify_study_taxonomy("MGYS00005116")
    if raw_taxonomy.empty:
        raise RuntimeError("empty response from MGnify")
    data_source = f" Real MGnify taxonomic data loaded ({raw_taxonomy['sample'].nunique()} samples)"
except Exception as e:
    data_source = f" MGnify live fetch unavailable ({str(e)[:70]}) — using literature-grounded simulated dataset"
    raw_taxonomy = None

print(data_source)


⚠️ MGnify live fetch unavailable (empty response from MGnify) — using literature-grounded simulated dataset


In [3]:
# Literature-grounded genus panel + known Healthy vs IBD-associated shift directions
# (directionality based on well-replicated findings in gut microbiome / IBD literature)
genus_panel = {
    "Faecalibacterium": -1.4, "Roseburia": -1.1, "Bacteroides": -0.3, "Prevotella": 0.2,
    "Ruminococcus": -0.6, "Akkermansia": -0.8, "Bifidobacterium": -0.5, "Lactobacillus": 0.1,
    "Escherichia": 1.6, "Shigella": 1.2, "Enterococcus": 1.0, "Fusobacterium": 1.3,
    "Clostridium": -0.4, "Blautia": -0.5, "Coprococcus": -0.7, "Alistipes": -0.2,
    "Parabacteroides": -0.3, "Veillonella": 0.9, "Streptococcus": 0.7, "Klebsiella": 1.1,
}

def simulate_microbiome_dataset(n_healthy=90, n_disease=70, genus_panel=genus_panel, seed=42):
    rng = np.random.default_rng(seed)
    genera = list(genus_panel.keys())
    rows, groups = [], []

    for _ in range(n_healthy):
        base = rng.dirichlet(np.ones(len(genera)) * 2)
        rows.append(base); groups.append("Healthy")
    for _ in range(n_disease):
        shift_weights = np.array([np.exp(genus_panel[g] * 0.8) for g in genera])
        base = rng.dirichlet(np.ones(len(genera)) * 2 * shift_weights)
        rows.append(base); groups.append("Disease (IBD-like)")

    abundance = pd.DataFrame(rows, columns=genera)
    abundance = abundance.div(abundance.sum(axis=1), axis=0)  # relative abundance, sums to 1
    abundance.index = [f"Sample_{i:03d}" for i in range(len(abundance))]
    metadata = pd.DataFrame({"sample": abundance.index, "group": groups})
    return abundance, metadata

if raw_taxonomy is None:
    abundance_df, metadata_df = simulate_microbiome_dataset()
else:
    # Pivot real MGnify data into a sample x taxon abundance matrix (best-effort parsing)
    pivot = raw_taxonomy.pivot_table(index="sample", columns="taxon", values="count", aggfunc="sum").fillna(0)
    abundance_df = pivot.div(pivot.sum(axis=1), axis=0)
    metadata_df = pd.DataFrame({"sample": abundance_df.index})
    metadata_df["group"] = np.random.default_rng(1).choice(["Healthy", "Disease (IBD-like)"], len(metadata_df))

print(f"Final abundance matrix: {abundance_df.shape[0]} samples x {abundance_df.shape[1]} taxa")
print(f"Group distribution:\n{metadata_df['group'].value_counts()}")
abundance_df.head()


Final abundance matrix: 160 samples x 20 taxa
Group distribution:
group
Healthy               90
Disease (IBD-like)    70
Name: count, dtype: int64


,Faecalibacterium,Roseburia,Bacteroides,Prevotella,Ruminococcus,Akkermansia,Bifidobacterium,Lactobacillus,Escherichia,Shigella,Enterococcus,Fusobacterium,Clostridium,Blautia,Coprococcus,Alistipes,Parabacteroides,Veillonella,Streptococcus,Klebsiella
Sample_000,0.050549,0.068516,0.044397,0.039753,0.074411,0.042371,0.056692,0.052909,0.074366,0.034779,0.091721,0.028336,0.059269,0.054572,0.026315,0.062710,0.036824,0.019641,0.068197,0.013673
Sample_001,0.028937,0.055689,0.093650,0.082706,0.063226,0.012364,0.034546,0.040840,0.023847,0.009212,0.057641,0.084480,0.038385,0.092856,0.015365,0.022594,0.056774,0.035876,0.079834,0.071179
Sample_002,0.039374,0.017472,0.035903,0.075145,0.085117,0.037381,0.009507,0.015104,0.071030,0.038868,0.039603,0.023125,0.022740,0.095343,0.072772,0.032098,0.055911,0.063018,0.149421,0.021068
Sample_003,0.052963,0.080609,0.117373,0.026107,0.032642,0.070769,0.010707,0.056676,0.156207,0.060874,0.004076,0.022133,0.026783,0.093059,0.039582,0.008206,0.043030,0.049555,0.029660,0.018992
Sample_004,0.135543,0.065581,0.069545,0.032185,0.020900,0.025039,0.010958,0.101623,0.033183,0.087105,0.026020,0.049916,0.041848,0.009855,0.109924,0.026651,0.013269,0.057764,0.036284,0.046809


## 3. Data Preprocessing

In [4]:
# Filter out extremely rare taxa (present in <5% of samples) — standard microbiome QC step
prevalence = (abundance_df > 0).mean(axis=0)
keep_taxa = prevalence[prevalence >= 0.05].index
abundance_df = abundance_df[keep_taxa]

print(f"Taxa retained after prevalence filtering: {abundance_df.shape[1]}")

# Also compute CLR (centered log-ratio) transform — standard for compositional microbiome data in ML
pseudo = abundance_df.replace(0, 1e-6)
clr_df = np.log(pseudo).sub(np.log(pseudo).mean(axis=1), axis=0)
print("CLR (centered log-ratio) transform computed for downstream ML use")


Taxa retained after prevalence filtering: 20
CLR (centered log-ratio) transform computed for downstream ML use


## 4. Taxonomic Composition Overview

In [5]:
mean_abundance = abundance_df.mean(axis=0).sort_values(ascending=False)
top_taxa = mean_abundance.head(12).index.tolist()

comp_df = abundance_df[top_taxa].copy()
comp_df["Other"] = 1 - comp_df.sum(axis=1)
comp_df["group"] = metadata_df.set_index("sample").loc[comp_df.index, "group"].values
comp_df["sample"] = comp_df.index

comp_melted = comp_df.melt(id_vars=["sample", "group"], var_name="taxon", value_name="relative_abundance")
comp_melted = comp_melted.sort_values(["group", "sample"])

fig = px.bar(comp_melted, x="sample", y="relative_abundance", color="taxon",
             title="Taxonomic Composition per Sample (Top 12 Genera)",
             template="plotly_white", color_discrete_sequence=px.colors.qualitative.Alphabet)
fig.update_layout(height=550, xaxis_showticklabels=False, legend=dict(font=dict(size=9)))
fig.show()


In [6]:
group_mean = abundance_df.groupby(metadata_df.set_index("sample")["group"]).mean()[top_taxa]

fig = px.imshow(group_mean.T.values, x=group_mean.index, y=top_taxa, color_continuous_scale="YlOrRd",
                 aspect="auto", title="Mean Relative Abundance by Group (Top Genera)",
                 labels=dict(color="Relative Abundance"))
fig.update_layout(height=500)
fig.show()


## 5. Alpha Diversity (Within-Sample Diversity)

In [7]:
def shannon_index(row):
    p = row[row > 0]
    return -np.sum(p * np.log(p))

def simpson_index(row):
    p = row[row > 0]
    return 1 - np.sum(p ** 2)

def observed_richness(row):
    return (row > 0).sum()

alpha_df = pd.DataFrame({
    "Shannon": abundance_df.apply(shannon_index, axis=1),
    "Simpson": abundance_df.apply(simpson_index, axis=1),
    "Richness": abundance_df.apply(observed_richness, axis=1),
})
alpha_df["group"] = metadata_df.set_index("sample").loc[alpha_df.index, "group"].values

fig = go.Figure()
from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=("Shannon Diversity", "Simpson Diversity", "Observed Richness"))

for i, metric in enumerate(["Shannon", "Simpson", "Richness"]):
    for grp in alpha_df["group"].unique():
        vals = alpha_df.loc[alpha_df["group"] == grp, metric]
        color = "#E63946" if "Disease" in grp else "#2E86AB"
        fig.add_trace(go.Box(y=vals, name=grp, marker_color=color, showlegend=(i == 0)), row=1, col=i+1)

fig.update_layout(height=450, title_text="Alpha Diversity — Healthy vs Disease")
fig.show()

for metric in ["Shannon", "Simpson", "Richness"]:
    h = alpha_df.loc[alpha_df["group"] == "Healthy", metric]
    d = alpha_df.loc[alpha_df["group"] != "Healthy", metric]
    stat, pval = mannwhitneyu(h, d)
    print(f"{metric}: Mann-Whitney U p-value = {pval:.4f} {'(significant)' if pval < 0.05 else ''}")


Shannon: Mann-Whitney U p-value = 0.0000 (significant)
Simpson: Mann-Whitney U p-value = 0.0000 (significant)
Richness: Mann-Whitney U p-value = 1.0000 


## 6. Beta Diversity & PCoA Ordination

In [8]:
bray_curtis = pdist(abundance_df.values, metric="braycurtis")
bc_matrix = squareform(bray_curtis)

try:
    from skbio import DistanceMatrix
    from skbio.stats.ordination import pcoa
    dm = DistanceMatrix(bc_matrix, ids=abundance_df.index)
    pcoa_result = pcoa(dm)
    pcoa_coords = pcoa_result.samples[["PC1", "PC2"]].values
    var_explained = pcoa_result.proportion_explained[:2].values * 100
except Exception:
    pca_bc = PCA(n_components=2)
    pcoa_coords = pca_bc.fit_transform(bc_matrix)
    var_explained = pca_bc.explained_variance_ratio_[:2] * 100

pcoa_df = pd.DataFrame(pcoa_coords, columns=["PC1", "PC2"], index=abundance_df.index)
pcoa_df["group"] = metadata_df.set_index("sample").loc[pcoa_df.index, "group"].values

fig = px.scatter(pcoa_df, x="PC1", y="PC2", color="group",
                  title=f"PCoA (Bray-Curtis) — PC1: {var_explained[0]:.1f}%, PC2: {var_explained[1]:.1f}%",
                  template="plotly_white", color_discrete_map={"Healthy": "#2E86AB", "Disease (IBD-like)": "#E63946"})
fig.update_traces(marker=dict(size=9, opacity=0.75, line=dict(width=0.4, color='white')))
fig.update_layout(height=550)
fig.show()

# PERMANOVA-style group separation check (simplified R2 via distance-based test)
from sklearn.metrics import silhouette_score as sil
group_labels = pcoa_df["group"].values
sep_score = sil(bc_matrix, group_labels, metric="precomputed")
print(f"Silhouette score (Bray-Curtis, Healthy vs Disease separation): {sep_score:.3f}")
print("(Higher = clearer community-level separation between groups)")


Silhouette score (Bray-Curtis, Healthy vs Disease separation): 0.200
(Higher = clearer community-level separation between groups)


## 7. Differential Abundance Testing

In [9]:
diff_results = []
healthy_mask = metadata_df.set_index("sample").loc[abundance_df.index, "group"] == "Healthy"

for taxon in abundance_df.columns:
    h_vals = abundance_df.loc[healthy_mask.values, taxon]
    d_vals = abundance_df.loc[~healthy_mask.values, taxon]
    if h_vals.sum() == 0 and d_vals.sum() == 0:
        continue
    stat, pval = mannwhitneyu(h_vals, d_vals, alternative="two-sided")
    log2fc = np.log2((d_vals.mean() + 1e-6) / (h_vals.mean() + 1e-6))
    diff_results.append({"taxon": taxon, "log2FC": log2fc, "pvalue": pval})

diff_df = pd.DataFrame(diff_results)
_, diff_df["padj"], _, _ = multipletests(diff_df["pvalue"], method="fdr_bh")
diff_df["neg_log10_padj"] = -np.log10(diff_df["padj"].replace(0, 1e-300))
diff_df["significant"] = diff_df["padj"] < 0.05
diff_df["direction"] = np.where(diff_df["significant"] & (diff_df["log2FC"] > 0), "Enriched in Disease",
                          np.where(diff_df["significant"] & (diff_df["log2FC"] < 0), "Enriched in Healthy", "Not Significant"))

fig = px.scatter(diff_df, x="log2FC", y="neg_log10_padj", color="direction", hover_name="taxon",
                  title="Differential Abundance — Volcano Plot (Healthy vs Disease)",
                  labels={"neg_log10_padj": "-log10(adjusted p-value)", "log2FC": "log2 Fold Change (Disease/Healthy)"},
                  template="plotly_white", color_discrete_map={"Enriched in Disease": "#E63946", "Enriched in Healthy": "#2E86AB", "Not Significant": "#B0B0B0"})
fig.add_hline(y=-np.log10(0.05), line_dash="dash", line_color="gray")
fig.update_traces(marker=dict(size=9, line=dict(width=0.5, color='white')))
fig.update_layout(height=550)
fig.show()

diff_df.sort_values("padj").head(10)


,taxon,log2FC,pvalue,padj,neg_log10_padj,significant,direction
0,Faecalibacterium,-2.168018,2.193483e-19,3.094192e-18,17.509453,True,Enriched in Healthy
8,Escherichia,1.191847,3.094192e-19,3.094192e-18,17.509453,True,Enriched in Disease
11,Fusobacterium,1.213425,3.619970e-18,2.413313e-17,16.617386,True,Enriched in Disease
1,Roseburia,-1.571279,3.256452e-13,1.628226e-12,11.788285,True,Enriched in Healthy
9,Shigella,0.886886,4.035148e-12,1.614059e-11,10.792081,True,Enriched in Disease
10,Enterococcus,0.872601,1.273319e-10,3.721300e-10,9.429305,True,Enriched in Disease
5,Akkermansia,-1.315629,1.302455e-10,3.721300e-10,9.429305,True,Enriched in Healthy
14,Coprococcus,-1.056851,2.282657e-09,5.706642e-09,8.243619,True,Enriched in Healthy
19,Klebsiella,0.743910,3.331179e-09,7.402620e-09,8.130615,True,Enriched in Disease
12,Clostridium,-0.996416,5.876372e-08,1.175274e-07,6.929861,True,Enriched in Healthy


## 8. Unsupervised Community Clustering (Enterotyping)

In [10]:
k_range = range(2, 6)
inertias, sil_scores = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(clr_df)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(clr_df, labels))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Elbow Method", "Silhouette Score"))
fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers', line=dict(color="#2E86AB")), row=1, col=1)
fig.add_trace(go.Scatter(x=list(k_range), y=sil_scores, mode='lines+markers', line=dict(color="#E63946")), row=1, col=2)
fig.update_layout(height=400, title_text="Optimal Cluster Number Selection", showlegend=False)
fig.show()

best_k = list(k_range)[np.argmax(sil_scores)]
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
community_types = km_final.fit_predict(clr_df)
pcoa_df["community_type"] = [f"Type {c+1}" for c in community_types]

fig2 = px.scatter(pcoa_df, x="PC1", y="PC2", color="community_type", symbol="group",
                   title=f"Community Types (k={best_k}) Overlaid on PCoA Space",
                   template="plotly_white", color_discrete_sequence=px.colors.qualitative.Set2)
fig2.update_traces(marker=dict(size=9))
fig2.update_layout(height=550)
fig2.show()

print(f"Optimal number of community types (enterotypes): {best_k}")
print(pd.crosstab(pcoa_df["community_type"], pcoa_df["group"]))


Optimal number of community types (enterotypes): 2
group           Disease (IBD-like)  Healthy
community_type                             
Type 1                           2       89
Type 2                          68        1


## 9. ML Classification — Healthy vs Disease

In [11]:
le = LabelEncoder()
y = le.fit_transform(metadata_df.set_index("sample").loc[abundance_df.index, "group"])
X = clr_df

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

microbiome_scaler = StandardScaler()
X_train_s = microbiome_scaler.fit_transform(X_train)
X_test_s = microbiome_scaler.transform(X_test)

microbiome_clf = RandomForestClassifier(n_estimators=400, random_state=42)
microbiome_clf.fit(X_train_s, y_train)

preds = microbiome_clf.predict(X_test_s)
probs = microbiome_clf.predict_proba(X_test_s)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, preds):.3f}  |  ROC-AUC: {roc_auc_score(y_test, probs):.3f}")
print(classification_report(y_test, preds, target_names=le.classes_))


Accuracy: 0.975  |  ROC-AUC: 1.000
                    precision    recall  f1-score   support

Disease (IBD-like)       1.00      0.94      0.97        18
           Healthy       0.96      1.00      0.98        22

          accuracy                           0.97        40
         macro avg       0.98      0.97      0.97        40
      weighted avg       0.98      0.97      0.97        40



In [ ]:
cm = confusion_matrix(y_test, preds)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues", x=le.classes_, y=le.classes_,
                 labels=dict(x="Predicted", y="Actual", color="Count"), title="Confusion Matrix")
fig.update_layout(height=450, width=500)
fig.show()

importances = pd.Series(microbiome_clf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
fig2 = px.bar(importances, orientation='h', title="Top 15 Discriminative Taxa (Feature Importance)",
              template="plotly_white", labels={"value": "Importance", "index": "Taxon"},
              color=importances.values, color_continuous_scale="Viridis")
fig2.update_layout(height=550, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig2.show()


## 10.  Runtime Prediction — Apna Microbiome Profile Daal Kar Predict Karein

Neeche key genera ki relative abundance (%) daalein — model predict karega ke profile **Healthy** jaisa hai ya **Disease-associated (IBD-like)**, saath mein ek radar chart bhi milega.


In [14]:
importances = pd.Series(microbiome_clf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
top_input_taxa = importances.head(10).index.tolist()

taxa_boxes = {}
taxa_widgets = []
defaults = abundance_df[top_input_taxa].mean().to_dict()
for taxon in top_input_taxa:
    slider = widgets.FloatSlider(value=round(defaults[taxon]*100, 2), min=0, max=40, step=0.5,
                                  description=taxon, style={'description_width': '130px'},
                                  layout=widgets.Layout(width='380px'), readout_format='.1f')
    taxa_boxes[taxon] = slider
    taxa_widgets.append(slider)

predict_btn = widgets.Button(description=" Profile Predict Karein", button_style='success',
                              layout=widgets.Layout(width='250px', height='42px'))
out = widgets.Output()

def render_microbiome_result(label, proba):
    color = "#E63946" if label == "Disease (IBD-like)" else "#2E86AB"
    emoji = "" if label == "Disease (IBD-like)" else ""
    conf = proba[1]*100 if label == le.classes_[1] else proba[0]*100
    html = f"""
    <div style="border:2px solid {color}; border-radius:12px; padding:18px; margin-top:12px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:21px; font-weight:700; color:{color};">{emoji} Predicted Profile: {label}</div>
        <div style="font-size:14px; margin-top:8px;">Confidence: <b>{conf:.1f}%</b></div>
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        user_vals = {t: taxa_boxes[t].value / 100 for t in top_input_taxa}
        other_taxa = [c for c in abundance_df.columns if c not in top_input_taxa]
        remaining = max(1 - sum(user_vals.values()), 1e-6)
        other_vals = {c: remaining / len(other_taxa) for c in other_taxa}
        full_profile = {**user_vals, **other_vals}

        profile_series = pd.Series(full_profile)[abundance_df.columns]
        profile_pseudo = profile_series.replace(0, 1e-6)
        profile_clr = np.log(profile_pseudo) - np.log(profile_pseudo).mean()

        row_scaled = microbiome_scaler.transform([profile_clr.values])
        pred = microbiome_clf.predict(row_scaled)[0]
        proba = microbiome_clf.predict_proba(row_scaled)[0]
        label = le.inverse_transform([pred])[0]
        render_microbiome_result(label, proba)

        radar_fig = go.Figure()
        radar_fig.add_trace(go.Scatterpolar(r=[user_vals[t]*100 for t in top_input_taxa] + [user_vals[top_input_taxa[0]]*100],
                                             theta=top_input_taxa + [top_input_taxa[0]], fill='toself',
                                             name="Your Profile", line_color=color if False else "#2E86AB"))
        healthy_avg = abundance_df.loc[metadata_df.set_index("sample")["group"] == "Healthy", top_input_taxa].mean() * 100
        radar_fig.add_trace(go.Scatterpolar(r=healthy_avg.tolist() + [healthy_avg.iloc[0]],
                                             theta=top_input_taxa + [top_input_taxa[0]], fill='toself',
                                             name="Healthy Average", opacity=0.5, line_color="#43AA8B"))
        radar_fig.update_layout(title="Your Profile vs Healthy Average", template="plotly_white", height=500)
        radar_fig.show()

predict_btn.on_click(on_predict)

display(widgets.HTML("<b style='font-size:15px;'>Key Genera — Relative Abundance (%)</b>"))
display(widgets.GridBox(taxa_widgets, layout=widgets.Layout(grid_template_columns="repeat(2, 400px)", grid_gap="6px")))
display(predict_btn)
display(out)

HTML(value="<b style='font-size:15px;'>Key Genera — Relative Abundance (%)</b>")

GridBox(children=(FloatSlider(value=8.46, description='Escherichia', layout=Layout(width='380px'), max=40.0, r…

Button(button_style='success', description=' Profile Predict Karein', layout=Layout(height='42px', width='250p…

Output()